# EA722 – Laboratório de Controle e Servomecanismo

## Experiência 7: Preditores e Controladores Não-Lineares

### Parte 2: Controle Neural Indireto (versão Keras / TensorFlow)

Universidade Estadual de Campinas – UNICAMP <br>
Faculdade de Engenharia Elétrica e de Computação – FEEC <br>

**Professores:** Fernando J. Von Zuben / Caíque Santos Lima <br>
**Grupo / Bancada:** T1, T2, R1, R2 ou E <br>
**Turma:** K, L, U ou V <br>
**Aluno(a):** , **RA:** ?????? <br>
**Aluno(a):** , **RA:** ?????? <br>
**Aluno(a):** , **RA:** ?????? <br>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    go = None


tf.random.set_seed(0)
np.random.seed(0)

##**Nonlinear cart-pole plant with theta=0 at the upright position.**


In [2]:
class InvertedPendulumCart:

    def __init__(
        self,
        cart_mass=1.0,
        pole_mass=0.1,
        half_pole_length=0.5,
        gravity=9.81,
        dt=0.02,
        force_limit=20.0,
    ):
        self.M = cart_mass
        self.m = pole_mass
        self.l = half_pole_length
        self.g = gravity
        self.dt = dt
        self.force_limit = force_limit

    def step(self, state, force):
        x, x_dot, theta, theta_dot = np.asarray(state, dtype=np.float64)
        force = float(np.clip(force, -self.force_limit, self.force_limit))

        total_mass = self.M + self.m
        sin_theta = np.sin(theta)
        cos_theta = np.cos(theta)

        temp = (force + self.m * self.l * theta_dot**2 * sin_theta) / total_mass
        theta_acc = (
            self.g * sin_theta - cos_theta * temp
        ) / (self.l * (4.0 / 3.0 - self.m * cos_theta**2 / total_mass))
        x_acc = temp - self.m * self.l * theta_acc * cos_theta / total_mass

        x_next = x + self.dt * x_dot
        x_dot_next = x_dot + self.dt * x_acc
        theta_next = theta + self.dt * theta_dot
        theta_dot_next = theta_dot + self.dt * theta_acc

        return np.asarray(
            [x_next, x_dot_next, theta_next, theta_dot_next],
            dtype=np.float32,
        )

##**Generating the training dataset.**


In [3]:
def generate_data(plant, rollouts=120, T=60):
    data = []

    for _ in range(rollouts):
        state = np.asarray(
            [
                np.random.uniform(-0.8, 0.8),
                np.random.uniform(-0.6, 0.6),
                np.random.uniform(-0.25, 0.25),
                np.random.uniform(-0.8, 0.8),
            ],
            dtype=np.float32,
        )

        for _ in range(T):
            force = np.random.uniform(-plant.force_limit, plant.force_limit)
            next_state = plant.step(state, force)
            data.append(np.concatenate([state, [force], next_state]))
            state = next_state

    data = np.asarray(data, dtype=np.float32)
    return data[:, :5], data[:, 5:]

##**Neural network plant emulator: definition and training steps.**


In [4]:
class PlantModel(keras.Model):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = keras.Sequential(
            [
                layers.Input(shape=(5,)),
                layers.Dense(hidden, activation="tanh"),
                layers.Dense(hidden, activation="tanh"),
                layers.Dense(4),
            ],
            name="cartpole_identifier",
        )

    def call(self, inputs, training=False):
        state, force = inputs
        x = tf.concat([state, force], axis=-1)
        delta_state = self.net(x, training=training)
        return state + delta_state


def train_plant_model(X, Y, epochs=501):
    model = PlantModel()
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    loss_fn = keras.losses.MeanSquaredError()

    X = tf.convert_to_tensor(X, dtype=tf.float32)
    Y = tf.convert_to_tensor(Y, dtype=tf.float32)
    state = X[:, :4]
    force = X[:, 4:5]

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            y_pred = model([state, force], training=True)
            loss = loss_fn(Y, y_pred)

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

        if epoch % 50 == 0:
            print(f"[Plant] Epoch {epoch}, Loss: {loss.numpy():.6f}")

    return model

##**Teacher controller: a linear quadratic regulator.**


In [5]:
def solve_discrete_lqr(A, B, Q, R, iterations=500):
    P = Q.copy()

    for _ in range(iterations):
        bt_p = B.T @ P
        P = A.T @ P @ A - A.T @ P @ B @ np.linalg.inv(R + bt_p @ B) @ bt_p @ A + Q

    return np.linalg.inv(R + B.T @ P @ B) @ (B.T @ P @ A)


def cartpole_lqr_gain(plant):
    M = plant.M
    m = plant.m
    l = plant.l
    g = plant.g
    dt = plant.dt

    A_c = np.asarray(
        [
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, -(m * g) / M, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, g * (M + m) / (l * M), 0.0],
        ],
        dtype=np.float64,
    )
    B_c = np.asarray([[0.0], [1.0 / M], [0.0], [-1.0 / (l * M)]], dtype=np.float64)

    A_d = np.eye(4) + dt * A_c
    B_d = dt * B_c
    Q = np.diag([6.0, 1.0, 80.0, 4.0])
    R = np.asarray([[0.08]])

    return solve_discrete_lqr(A_d, B_d, Q, R)


def state_feedback_control(state, K, force_limit):
    """Linear full-state feedback: u = -Kx."""
    force = -float((K @ np.asarray(state, dtype=np.float64).reshape(4, 1))[0, 0])
    return np.clip(force, -force_limit, force_limit)

##**Neural network plant controller: definition and training steps to emulate a linear state feedback.**


In [6]:
class Controller(keras.Model):
    def __init__(self, force_limit=20.0, hidden=64):
        super().__init__()
        self.force_limit = force_limit
        self.output_gain = tf.Variable(1.0, trainable=False, dtype=tf.float32)
        self.output_offset = tf.Variable(0.0, trainable=False, dtype=tf.float32)
        self.net = keras.Sequential(
            [
                layers.Input(shape=(4,)),
                layers.Dense(hidden, activation="tanh"),
                layers.Dense(hidden, activation="tanh"),
                layers.Dense(1),
            ],
            name="cartpole_neural_controller",
        )

    def call(self, state, training=False):
        raw_force = self.net(state, training=training)
        force = self.output_gain * raw_force + self.output_offset
        return self.force_limit * tf.tanh(force / self.force_limit)

def train_controller_from_linear_feedback(plant, K, epochs=501, samples=8000):
    controller = Controller(force_limit=plant.force_limit)
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    loss_fn = keras.losses.MeanSquaredError()

    states = np.column_stack(
        [
            np.random.uniform(-1.0, 1.0, samples),
            np.random.uniform(-1.0, 1.0, samples),
            np.random.uniform(-0.35, 0.35, samples),
            np.random.uniform(-1.5, 1.5, samples),
        ]
    ).astype(np.float32)
    forces = np.asarray(
        [state_feedback_control(state, K, plant.force_limit) for state in states],
        dtype=np.float32,
    ).reshape(-1, 1)

    states = tf.convert_to_tensor(states, dtype=tf.float32)
    forces = tf.convert_to_tensor(forces, dtype=tf.float32)

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            force_pred = controller(states, training=True)
            loss = loss_fn(forces, force_pred)

        gradients = tape.gradient(loss, controller.trainable_variables)
        optimizer.apply_gradients(zip(gradients, controller.trainable_variables))

        if epoch % 50 == 0:
            print(f"[Linear feedback emulation] Epoch {epoch}, Loss: {loss.numpy():.6f}")

    return controller

def calibrate_controller_offset(controller):
    zero_state = tf.zeros((1, 4), dtype=tf.float32)
    force_at_origin = controller(zero_state, training=False).numpy()[0, 0]
    controller.output_offset.assign_add(-force_at_origin)
    return {
        "force_at_origin_before": float(force_at_origin),
        "offset": float(controller.output_offset.numpy()),
    }

##**Neural network controller acting on the neural network plant emulator.**


In [7]:
def simulate_model(controller, model, initial_state=None, T=160):
    if initial_state is None:
        initial_state = [0.0, 0.0, np.deg2rad(15.0), 0.0]

    state = tf.convert_to_tensor([initial_state], dtype=tf.float32)
    states = []
    forces = []

    for _ in range(T):
        force = controller(state, training=True)
        state = model([state, force], training=False)
        states.append(state)
        forces.append(force)

    return tf.concat(states, axis=0), tf.concat(forces, axis=0)

##**Neural network controller refinement using indirect control.**


In [8]:
def refine_controller_with_model(model, controller, epochs=201):
    optimizer = keras.optimizers.Adam(learning_rate=0.0005)
    model.trainable = False
    loss_history = []

    initial_conditions = tf.constant(
        [
            [0.0, 0.0, np.deg2rad(15.0), 0.0],
            [0.2, 0.0, np.deg2rad(10.0), 0.0],
            [-0.2, 0.0, np.deg2rad(-10.0), 0.0],
        ],
        dtype=tf.float32,
    )

    Q1 = tf.constant([6.0, 1.0, 80.0, 4.0], dtype=tf.float32)
    R1 = 0.08

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            total_loss = 0.0
            for i in range(initial_conditions.shape[0]):
                states, forces = simulate_model(
                    controller,
                    model,
                    initial_state=initial_conditions[i].numpy(),
                    T=100,
                )
                state_loss = tf.reduce_mean(tf.reduce_sum(Q1 * tf.square(states), axis=1))
                force_loss = R1 * tf.reduce_mean(tf.square(forces))
                total_loss += state_loss + force_loss

            loss = total_loss / float(initial_conditions.shape[0])

        gradients = tape.gradient(loss, controller.trainable_variables)
        optimizer.apply_gradients(zip(gradients, controller.trainable_variables))
        loss_history.append(float(loss.numpy()))

        if epoch % 50 == 0:
            print(f"[Control refinement] Epoch {epoch}, Loss: {loss.numpy():.6f}")

    return controller, loss_history

##**Simulating neural and linear controllers.**


In [9]:
def simulate_neural_controller(controller, plant, initial_state=None, T=240):
    if initial_state is None:
        initial_state = [0.0, 0.0, np.deg2rad(15.0), 0.0]

    state = np.asarray(initial_state, dtype=np.float32)
    states = []
    forces = []

    for _ in range(T):
        force = controller(
            tf.constant([state], dtype=tf.float32),
            training=False,
        ).numpy()[0, 0]
        state = plant.step(state, force)
        states.append(state)
        forces.append(force)

    return np.asarray(states), np.asarray(forces)


def simulate_linear_controller(K, plant, initial_state=None, T=240):
    if initial_state is None:
        initial_state = [0.0, 0.0, np.deg2rad(15.0), 0.0]

    state = np.asarray(initial_state, dtype=np.float32)
    states = []
    forces = []

    for _ in range(T):
        force = state_feedback_control(state, K, plant.force_limit)
        state = plant.step(state, force)
        states.append(state)
        forces.append(force)

    return np.asarray(states), np.asarray(forces)

##**Comparative analysis.**


In [10]:
def plot_state_comparison1(neural_states, linear_states, dt, title):
    time = np.arange(len(neural_states)) * dt

    if go is not None:
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=time, y=neural_states[:, 0], name="x neural [m]"))
        fig.add_trace(go.Scatter(x=time, y=linear_states[:, 0], name="x linear [m]"))
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(title=title, template="plotly_white", xaxis_title="time [s]")
        fig.show()
        return

    plt.figure(figsize=(12, 8))
    plt.plot(time, neural_states[:, 0], label="x neural [m]")
    plt.plot(time, linear_states[:, 0], label="x linear [m]")
    plt.axhline(0.0, linestyle="--", color="black")
    plt.title(title)
    plt.xlabel("time [s]")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_state_comparison2(neural_states, linear_states, dt, title):
    time = np.arange(len(neural_states)) * dt
    neural_theta_deg = np.rad2deg(neural_states[:, 2])
    linear_theta_deg = np.rad2deg(linear_states[:, 2])

    if go is not None:
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=time, y=neural_theta_deg, name="theta neural [deg]"))
        fig.add_trace(go.Scatter(x=time, y=linear_theta_deg, name="theta linear [deg]"))
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(title=title, template="plotly_white", xaxis_title="time [s]")
        fig.show()
        return

    plt.figure(figsize=(12, 8))
    plt.plot(time, neural_theta_deg, label="theta neural [deg]")
    plt.plot(time, linear_theta_deg, label="theta linear [deg]")
    plt.axhline(0.0, linestyle="--", color="black")
    plt.title(title)
    plt.xlabel("time [s]")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_control_comparison(neural_forces, linear_forces, dt, title):
    time = np.arange(len(neural_forces)) * dt

    if go is not None:
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=time, y=neural_forces, name="u neural [N]"))
        fig.add_trace(go.Scatter(x=time, y=linear_forces, name="u linear [N]"))
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(title=title, template="plotly_white", xaxis_title="time [s]")
        fig.show()
        return

    plt.figure(figsize=(12, 8))
    plt.plot(time, neural_forces, label="u neural [N]")
    plt.plot(time, linear_forces, label="u linear [N]")
    plt.axhline(0.0, linestyle="--", color="black")
    plt.title(title)
    plt.xlabel("time [s]")
    plt.ylabel("force [N]")
    plt.legend()
    plt.grid(True)
    plt.show()


def performance_metrics(states, forces):
    x = states[:, 0]
    theta = states[:, 2]

    return {
        "control_energy": float(np.sum(np.square(forces))),
        "final_x": float(x[-1]),
        "final_theta_deg": float(np.rad2deg(theta[-1])),
        "steady_state_position_error": float(abs(x[-1])),
        "steady_state_angle_error_deg": float(abs(np.rad2deg(theta[-1]))),
        "max_abs_x": float(np.max(np.abs(x))),
        "max_abs_theta_deg": float(np.max(np.abs(np.rad2deg(theta)))),
    }

##**Defining the full pipeline.**


In [11]:
def run_full_pipeline(show_plots=True, refine_epochs=201):
    plant = InvertedPendulumCart()
    initial_state = [0.0, 0.0, np.deg2rad(15.0), 0.0]

    X, Y = generate_data(plant)
    model = train_plant_model(X, Y)

    K = cartpole_lqr_gain(plant)
    controller = train_controller_from_linear_feedback(plant, K)
    if refine_epochs > 0:
        controller, loss_hist = refine_controller_with_model(
            model,
            controller,
            epochs=refine_epochs,
        )
    else:
        loss_hist = []

    calibration = calibrate_controller_offset(controller)
    print("Controller calibration:", calibration)
    print("Linear state feedback gain K:", K)

    neural_states, neural_forces = simulate_neural_controller(
        controller,
        plant,
        initial_state=initial_state,
    )
    linear_states, linear_forces = simulate_linear_controller(
        K,
        plant,
        initial_state=initial_state,
    )

    neural_metrics = performance_metrics(neural_states, neural_forces)
    linear_metrics = performance_metrics(linear_states, linear_forces)

    print("Neural controller:", neural_metrics)
    print("Linear state feedback:", linear_metrics)

    if show_plots:
        plot_state_comparison1(
            neural_states,
            linear_states,
            plant.dt,
            "Cart Position Response: Neural x Linear Feedback",
        )
        plot_state_comparison2(
            neural_states,
            linear_states,
            plant.dt,
            "Pole Position Response: Neural x Linear Feedback",
        )
        plot_control_comparison(
            neural_forces,
            linear_forces,
            plant.dt,
            "Control Action: Neural x Linear Feedback",
        )

    return {
        "plant": plant,
        "model": model,
        "controller": controller,
        "linear_gain": K,
        "loss_history": loss_hist,
        "calibration": calibration,
        "initial_state": initial_state,
        "neural_states": neural_states,
        "neural_forces": neural_forces,
        "linear_states": linear_states,
        "linear_forces": linear_forces,
        "neural_metrics": neural_metrics,
        "linear_metrics": linear_metrics,
    }

##**Running the full pipeline.**


In [ ]:
results = run_full_pipeline(show_plots=True)


<font color="green">
Atividade 2.1 <br>
Explique o motivo pelo qual há três etapas de treinamento envolvidas neste projeto de controle neural indireto.
</font>

Resposta:

<font color="green">
Atividade 2.2 <br>
Quais são as vantagens e desvantagens ao se adotar um controle não-linear, quando se compara com a solução de controle linear?
</font>

Resposta: